# BTCUSDT — exploration and prediction

In [1]:
# Setup
import sys
sys.path.insert(0, ".")          # so `import research` finds research.py
import research as R
import features as F
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

ROOT = "./data"
pd.set_option("display.width", 140)

In [2]:
inv = R.inventory(ROOT)
print(inv['stream'].value_counts())
inv.info()
inv.head()



stream
agg_trade      224
book_ticker    224
depth          224
mark_price     112
force_order    109
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 893 entries, 0 to 892
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   venue   893 non-null    str    
 1   symbol  893 non-null    str    
 2   stream  893 non-null    str    
 3   date    893 non-null    str    
 4   hour    893 non-null    str    
 5   rows    893 non-null    int64  
 6   mb      893 non-null    float64
dtypes: float64(1), int64(1), str(5)
memory usage: 78.9 KB


,venue,symbol,stream,date,hour,rows,mb
0,futures,BTCUSDT,agg_trade,2026-08-06,20,4612,0.2
1,futures,BTCUSDT,agg_trade,2026-08-06,21,16553,0.6
2,futures,BTCUSDT,agg_trade,2026-08-06,22,13878,0.5
3,futures,BTCUSDT,agg_trade,2026-08-06,23,16573,0.6
4,futures,BTCUSDT,agg_trade,2026-08-07,00,22265,0.8


In [3]:
# Cell 4 — spot BBO (the target instrument) on the mandated 1-second grid
bbo = R.add_mid(R.load(ROOT, "spot", stream="book_ticker", last_hours=80))
print(f"{len(bbo):,} source rows, "
      f"{(bbo.recv_ns.max()-bbo.recv_ns.min())/1e9/3600:.2f} h span, "
      f"{R.contiguous_blocks(bbo).nunique()} contiguous block(s)")

grid = R.grid_labels(bbo, horizon_s=5.0, freq_s=1.0)
R.grid_report(grid)
grid.head(10)
grid.info()

14,072,092 source rows, 79.95 h span, 5 contiguous block(s)
grid points          : 287,831
  no prior obs / stale: 198 (0.07%)
  dropped, incl. tail : 380 (0.13%)
  usable labels       : 287,451 (99.87%)
staleness of sampled quote: p50=37ms p99=1039ms
y_bps: std=0.606 min=-15.15 max=26.14 |y|<0.1bps=80.8%
<class 'pandas.DataFrame'>
RangeIndex: 287831 entries, 0 to 287830
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   t_ns          287831 non-null  int64              
 1   mid           287633 non-null  float64            
 2   mid_fwd       287451 non-null  float64            
 3   staleness_ns  287831 non-null  int64              
 4   sess          287633 non-null  str                
 5   y_bps         287451 non-null  float64            
 6   t             287831 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), float64(3), int64(2), str(1)
memory usage: 18.7 MB


In [4]:
fut = R.add_mid(R.load(ROOT, "futures", stream="book_ticker", last_hours=80))
tr_spot = R.load(ROOT, "spot",    stream="agg_trade",  last_hours=80)
tr_fut  = R.load(ROOT, "futures", stream="agg_trade",  last_hours=80)
mark    = R.load(ROOT, "futures", stream="mark_price", last_hours=80)

In [5]:
bad = grid[grid.staleness_ns < 0]
print(f"{len(bad):,} unusable grid points")
if len(bad):
    blk = (bad.t_ns.diff() > 2e9).cumsum()
    runs = bad.groupby(blk).t_ns.agg(['min','max','count'])
    runs['minutes'] = (runs['max']-runs['min'])/1e9/60
    print(runs.assign(start=pd.to_datetime(runs['min'],unit='ns',utc=True))
              [['start','count','minutes']].to_string())

198 unusable grid points
                         start  count   minutes
t_ns                                           
0    2026-08-08 08:04:33+00:00      1  0.000000
1    2026-08-08 08:11:00+00:00      1  0.000000
2    2026-08-08 08:58:47+00:00      1  0.000000
3    2026-08-08 09:00:13+00:00      1  0.000000
4    2026-08-08 09:02:47+00:00      1  0.000000
5    2026-08-08 09:47:33+00:00      1  0.000000
6    2026-08-08 10:09:18+00:00      1  0.000000
7    2026-08-08 10:10:30+00:00      1  0.000000
8    2026-08-08 10:10:45+00:00      1  0.000000
9    2026-08-08 10:10:51+00:00      1  0.000000
10   2026-08-08 10:15:30+00:00      1  0.000000
11   2026-08-08 10:20:40+00:00      1  0.000000
12   2026-08-08 10:21:05+00:00      1  0.000000
13   2026-08-08 10:22:41+00:00      1  0.000000
14   2026-08-08 10:28:13+00:00      1  0.000000
15   2026-08-08 10:28:23+00:00      1  0.000000
16   2026-08-08 12:13:36+00:00      1  0.000000
17   2026-08-08 12:37:00+00:00      1  0.000000
18   2026-08-08

In [6]:
grid = R.grid_labels(bbo, horizon_s=5.0, freq_s=1.0)
X = F.build(grid, bbo, fut, tr_spot, tr_fut, mark)
F.assert_causal(X)

causality OK — every joined observation is at or before its grid point


In [7]:
print(X.columns)

Index(['t_ns', 'mid', 'mid_fwd', 'staleness_ns', 'sess', 'y_bps', 't', 'spot_age_s', 'spot_mid', 'spot_spread_bps', 'spot_imbalance',
       'spot_wmid_dev_bps', 'fut_age_s', 'fut_mid', 'fut_spread_bps', 'fut_imbalance', 'fut_wmid_dev_bps', 'spot_signed_vol_1s',
       'spot_n_trades_1s', 'spot_ofi_1s', 'spot_signed_vol_5s', 'spot_n_trades_5s', 'spot_ofi_5s', 'spot_signed_vol_60s',
       'spot_n_trades_60s', 'spot_ofi_60s', 'fut_signed_vol_1s', 'fut_n_trades_1s', 'fut_ofi_1s', 'fut_signed_vol_5s', 'fut_n_trades_5s',
       'fut_ofi_5s', 'fut_signed_vol_60s', 'fut_n_trades_60s', 'fut_ofi_60s', 'rv_10s_bps', 'rv_60s_bps', 'ret_1s_bps', 'basis_bps',
       'funding_rate'],
      dtype='str')


In [8]:
print(F.FEATURES)
d = X.dropna(subset=F.FEATURES + ["y_bps"])
Xf, y = d[F.FEATURES], d["y_bps"]

['spot_imbalance', 'spot_spread_bps', 'spot_wmid_dev_bps', 'fut_imbalance', 'fut_spread_bps', 'fut_wmid_dev_bps', 'basis_bps', 'spot_ofi_1s', 'spot_ofi_5s', 'fut_ofi_1s', 'fut_ofi_5s', 'fut_ofi_60s', 'fut_n_trades_5s', 'spot_n_trades_5s', 'rv_10s_bps', 'rv_60s_bps', 'ret_1s_bps']


In [9]:
print(y.value_counts())

y_bps
 0.000000    222592
 0.307659        12
-0.307461        12
-0.615839        11
 1.230845        11
              ...  
 3.344791         1
 0.365108         1
 0.055080         1
 0.258084         1
 0.314728         1
Name: count, Length: 29792, dtype: int64


In [14]:
import splits as S

d = X.dropna(subset=F.FEATURES + ["y_bps"])
dev, hold = S.frozen_holdout(d, frac=0.20, horizon_s=5.0, embargo_s=60.0)
# hold goes in a drawer until the model is final

print(S.describe_splits(dev, n_splits=5).to_string(index=False))


frozen holdout at 2026-08-10 23:58:44+00:00
  develop: 229,746 rows   holdout: 57,453 rows
  purged+embargoed at the boundary: 64 rows
 fold  n_train  n_val                 train_end                 val_start  gap_s
    0    38227  38291 2026-08-08 18:39:27+00:00 2026-08-08 18:40:32+00:00   65.0
    1    76518  38291 2026-08-09 05:19:55+00:00 2026-08-09 05:21:00+00:00   65.0
    2   114809  38291 2026-08-09 16:00:02+00:00 2026-08-09 16:01:07+00:00   65.0
    3   153100  38291 2026-08-10 02:39:43+00:00 2026-08-10 02:40:48+00:00   65.0
    4   191391  38291 2026-08-10 13:18:09+00:00 2026-08-10 13:19:14+00:00   65.0


In [15]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

def make_model():
    return make_pipeline(StandardScaler(), Ridge(alpha=1.0))

In [16]:
rows = []
for i, (tr, va) in enumerate(S.purged_walk_forward(dev, n_splits=5)):
    m = make_model()
    m.fit(dev.loc[tr, F.FEATURES], dev.loc[tr, "y_bps"])
    pred = m.predict(dev.loc[va, F.FEATURES])
    rows.append({"fold": i, **S.evaluate(dev.loc[va, "y_bps"], pred, theta=1.0)})

cv = pd.DataFrame(rows)
print(cv.round(5).to_string(index=False))
print("\nmean:", cv.r2_zero.mean().round(5), " std:", cv.r2_zero.std().round(5))

 fold  r2_zero      ic  ic_spearman  dir_acc_gt_1.0bp  n_gt_1.0bp  rmse_bps     n
    0  0.01594 0.15516      0.31652           0.77255         510   0.25889 38291
    1  0.03484 0.18444      0.35927           0.83247         579   0.28441 38291
    2  0.03756 0.22888      0.43652           0.79421        3246   0.72708 38291
    3  0.07814 0.28932      0.47513           0.80791        3337   0.59196 38291
    4  0.05668 0.25007      0.40605           0.76048        5607   0.90686 38291

mean: 0.04463  std: 0.02365
